In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from math import radians, cos, sin, asin, sqrt
from numpy import log

# Set pandas options for nice tables with only 2 decimals : 
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Find the true distance between two points in spherical coordinates.
    """

    R = 6372.8*1000 # For Earth radius in kilometers use 6372.8 km
    dLat = radians(lat2 - lat1)
    dLon = radians(lon2 - lon1)
    lat1 = radians(lat1)
    lat2 = radians(lat2)

    a = sin(dLat/2)**2 + cos(lat1)*cos(lat2)*sin(dLon/2)**2
    c = 2*asin(sqrt(a))

    return R * c

def bearing(lat1,lon1,lat2,lon2) :
    """
    Returns the bearing angle (North = 0), positive counter clockwise. Converts data to rad and outputs 
    bearing angle in degrees. 
    """
        
    x1_rad = radians(lon1)
    x2_rad = radians(lon2)
    
    y1_rad = radians(lat1)
    y2_rad = radians(lat2)
    
    y = sin(x2_rad-x1_rad) * cos(y2_rad)
    x = cos(y1_rad) * sin(y2_rad) - sin(y1_rad) * cos(y2_rad) * cos(x2_rad-x1_rad)
    
    theta = (-np.arctan2(y,x))*(360/(2*np.pi))
    return theta

In [4]:
atlantic_storms = pd.read_csv('atlantic_storms.csv')
display(atlantic_storms.head(5))

,index,id,name,date,record_identifier,status_of_system,latitude,longitude,maximum_sustained_wind_knots,maximum_pressure,...,34_kt_sw,34_kt_nw,50_kt_ne,50_kt_se,50_kt_sw,50_kt_nw,64_kt_ne,64_kt_se,64_kt_sw,64_kt_nw
0,0,AL011851,UNNAMED,1851-06-25 00:00:00,NaN,HU,28.00,-94.80,80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,AL011851,UNNAMED,1851-06-25 06:00:00,NaN,HU,28.00,-95.40,80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,AL011851,UNNAMED,1851-06-25 12:00:00,NaN,HU,28.00,-96.00,80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,AL011851,UNNAMED,1851-06-25 18:00:00,NaN,HU,28.10,-96.50,80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,AL011851,UNNAMED,1851-06-25 21:00:00,L,HU,28.20,-96.80,80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Defining new parameters to individual storms : 

In [5]:
storms_df = {}
storm_ids = atlantic_storms['id'].unique()

for i in storm_ids : 
    storms = atlantic_storms.loc[atlantic_storms['id'] == i]
    storms_df[i] = pd.DataFrame(storms)
    
# To get a quick look in table format: 
Visualize = False
# Number of subsets to look at : 
n_views = 5

if Visualize == True : 
    for dataframe in list(storms_df.values())[0:n_views]: 
        display(dataframe)

In [6]:
Calc_values_storms = pd.DataFrame()

for dataframe in list(storms_df.values()) : 
#for dataframe in list(storms_df.values())[0:5] :
    
    # Get general info : 
    x = np.array(dataframe['longitude'])
    y = np.array(dataframe['latitude'])
    storm_name = dataframe['name']
    storm_id = dataframe['id']
    ind = dataframe.index[0] # get where the first value of "dataframe" is relative to whole atlantic storms df.
    
    # Time in between measurements : 
    time_delta_gen = pd.to_datetime(dataframe['date'].astype(str)).diff()    
    time_delta_sec = pd.to_timedelta(time_delta_gen, unit = 's').dt.total_seconds()
    
    # Get the speed and heading : 
    speed = np.array([np.nan])                  # Ensure no speed calculation is performed on 1st point.
    heading = np.array([np.nan])
    
    for i in range(dataframe.shape[0]-1) :      # Do not overshoot the last value  
        
        # Initial coordinates :
        long1 = x[i]
        lat1 = y[i]
        
        # End coordinates : 
        long2 = x[i+1]
        lat2 = y[i+1]
        
        # Take the speed (and distance) with haversine model of earth : 
        speed = np.append(speed,(haversine(lat1, long1, lat2, long2)/(time_delta_sec[ind+i+1])))
                
        # Take the heading angle, considering the earth's geometry :
        heading = np.append(heading,(bearing(lat1,long1,lat2,long2)))   
    
    # Previous heading direction : 
    thetas_i_1 = np.insert(heading,0,np.nan) # Create an offset
    thetas_i1 = thetas_i_1[:-1] # Pop the last one
    
    # Angular change in direction : 
    delta_theta_offset = np.diff(heading)
    delta_theta = np.insert(delta_theta_offset,len(delta_theta_offset),0)
    delta_theta_v2 = delta_theta   
    delta_theta_v2 = np.where(delta_theta_v2< -180,delta_theta_v2+360, delta_theta_v2)
    delta_theta_v2= np.where(delta_theta_v2> 180,delta_theta_v2-360, delta_theta_v2)
    
    theta_ip1 = heading + delta_theta
    
    # Change in speed : 
    
    speed_offset = speed[1:]
    ln_speed_offset = np.log(speed_offset, out =np.zeros_like(speed_offset), where = (speed_offset!=0))
    
    ln_speed_offset = np.insert(ln_speed_offset,len(speed_offset),0)    
    speed_offset = np.insert(speed[1:],len(speed)-1,0)     
    ln_speed = np.log(speed, out =np.zeros_like(speed), where = (speed!=0))    

    delta_ln_speed = ln_speed_offset - ln_speed
    delta_speed = speed_offset - speed
    
    # Overwrite the 1st dataset to provide initial conditions : 
    # Note : the initial conditions means the 1st dataset should not be used to measure
    # speed and angular variations.
    # Same applies for final datasets (NaN values are set equal to 0). 
    
    if dataframe.shape[0] > 1 : 
    
        speed[0] = speed[1]
        heading[0] = heading[1]
        thetas_i1[1] = heading[0]
    
    sub_df = pd.DataFrame()
    sub_df['id'] = storm_id
    sub_df['name'] = storm_name
    sub_df['time_delta_s'] = time_delta_sec
    sub_df['latitude'] = y
    sub_df['longitude'] = x
    sub_df['Vi'] = speed
    sub_df['V_ip1'] = speed_offset
    sub_df['d(V)'] = delta_speed
    sub_df['ln_V'] = ln_speed
    sub_df['ln_Vip1'] = ln_speed_offset
    sub_df['d(ln_V)'] = delta_ln_speed
    sub_df['theta'] = heading
    sub_df['theta_im1'] = thetas_i1
    sub_df['delta_theta'] = delta_theta
    sub_df['delta_theta_v2'] = delta_theta_v2
    sub_df['theta_ip1'] = theta_ip1

    # Make it into a nice df (optional) : 
    view_df = False
    if view_df == True : 
        display(sub_df)
        
    Calc_values_storms= pd.concat([Calc_values_storms,sub_df], sort= False)
    
display(Calc_values_storms)

,id,name,time_delta_s,latitude,longitude,Vi,V_ip1,d(V),ln_V,ln_Vip1,d(ln_V),theta,theta_im1,delta_theta,delta_theta_v2,theta_ip1
0,AL011851,UNNAMED,NaN,28.00,-94.80,2.73,2.73,NaN,NaN,1.00,NaN,89.86,NaN,NaN,NaN,NaN
1,AL011851,UNNAMED,21600.00,28.00,-95.40,2.73,2.73,-0.00,1.00,1.00,-0.00,89.86,89.86,-0.00,-0.00,89.86
2,AL011851,UNNAMED,21600.00,28.00,-96.00,2.73,2.33,-0.40,1.00,0.85,-0.16,89.86,89.86,-12.75,-12.75,77.11
3,AL011851,UNNAMED,21600.00,28.10,-96.50,2.33,2.91,0.58,0.85,1.07,0.22,77.11,89.86,-7.89,-7.89,69.22
4,AL011851,UNNAMED,10800.00,28.20,-96.80,2.91,1.82,-1.10,1.07,0.60,-0.47,69.22,77.11,20.73,20.73,89.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52686,AL302020,THETA,21600.00,31.40,-18.30,1.35,1.67,0.32,0.30,0.51,0.21,-139.50,-106.26,87.60,87.60,-51.90
52687,AL302020,THETA,21600.00,31.60,-18.00,1.67,2.61,0.94,0.51,0.96,0.45,-51.90,-139.50,42.29,42.29,-9.62
52688,AL302020,THETA,21600.00,32.10,-17.90,2.61,3.35,0.74,0.96,1.21,0.25,-9.62,-51.90,32.42,32.42,22.81
52689,AL302020,THETA,21600.00,32.70,-18.20,3.35,4.00,0.64,1.21,1.39,0.18,22.81,-9.62,2.68,2.68,25.48


In [7]:
# Merge everything back together : 
edited_atlantic_storms_df = pd.concat([atlantic_storms,Calc_values_storms],axis = 1)
edited_atlantic_storms_df = edited_atlantic_storms_df.T.drop_duplicates().T
display(edited_atlantic_storms_df)

,index,id,name,date,record_identifier,status_of_system,latitude,longitude,maximum_sustained_wind_knots,maximum_pressure,...,V_ip1,d(V),ln_V,ln_Vip1,d(ln_V),theta,theta_im1,delta_theta,delta_theta_v2,theta_ip1
0,0,AL011851,UNNAMED,1851-06-25 00:00:00,NaN,HU,28.00,-94.80,80,NaN,...,2.73,NaN,NaN,1.00,NaN,89.86,NaN,NaN,NaN,NaN
1,1,AL011851,UNNAMED,1851-06-25 06:00:00,NaN,HU,28.00,-95.40,80,NaN,...,2.73,-0.00,1.00,1.00,-0.00,89.86,89.86,-0.00,-0.00,89.86
2,2,AL011851,UNNAMED,1851-06-25 12:00:00,NaN,HU,28.00,-96.00,80,NaN,...,2.33,-0.40,1.00,0.85,-0.16,89.86,89.86,-12.75,-12.75,77.11
3,3,AL011851,UNNAMED,1851-06-25 18:00:00,NaN,HU,28.10,-96.50,80,NaN,...,2.91,0.58,0.85,1.07,0.22,77.11,89.86,-7.89,-7.89,69.22
4,4,AL011851,UNNAMED,1851-06-25 21:00:00,L,HU,28.20,-96.80,80,NaN,...,1.82,-1.10,1.07,0.60,-0.47,69.22,77.11,20.73,20.73,89.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52686,52686,AL302020,THETA,2020-11-15 12:00:00,NaN,LO,31.40,-18.30,30,1009.00,...,1.67,0.32,0.30,0.51,0.21,-139.50,-106.26,87.60,87.60,-51.90
52687,52687,AL302020,THETA,2020-11-15 18:00:00,NaN,LO,31.60,-18.00,30,1012.00,...,2.61,0.94,0.51,0.96,0.45,-51.90,-139.50,42.29,42.29,-9.62
52688,52688,AL302020,THETA,2020-11-16 00:00:00,NaN,LO,32.10,-17.90,30,1013.00,...,3.35,0.74,0.96,1.21,0.25,-9.62,-51.90,32.42,32.42,22.81
52689,52689,AL302020,THETA,2020-11-16 06:00:00,NaN,LO,32.70,-18.20,25,1014.00,...,4.00,0.64,1.21,1.39,0.18,22.81,-9.62,2.68,2.68,25.48


In [8]:
edited_atlantic_storms_df['E_or_W'] = ['W' if x > 0 else 'E' for x in edited_atlantic_storms_df['theta']]
display(edited_atlantic_storms_df)

,index,id,name,date,record_identifier,status_of_system,latitude,longitude,maximum_sustained_wind_knots,maximum_pressure,...,d(V),ln_V,ln_Vip1,d(ln_V),theta,theta_im1,delta_theta,delta_theta_v2,theta_ip1,E_or_W
0,0,AL011851,UNNAMED,1851-06-25 00:00:00,NaN,HU,28.00,-94.80,80,NaN,...,NaN,NaN,1.00,NaN,89.86,NaN,NaN,NaN,NaN,W
1,1,AL011851,UNNAMED,1851-06-25 06:00:00,NaN,HU,28.00,-95.40,80,NaN,...,-0.00,1.00,1.00,-0.00,89.86,89.86,-0.00,-0.00,89.86,W
2,2,AL011851,UNNAMED,1851-06-25 12:00:00,NaN,HU,28.00,-96.00,80,NaN,...,-0.40,1.00,0.85,-0.16,89.86,89.86,-12.75,-12.75,77.11,W
3,3,AL011851,UNNAMED,1851-06-25 18:00:00,NaN,HU,28.10,-96.50,80,NaN,...,0.58,0.85,1.07,0.22,77.11,89.86,-7.89,-7.89,69.22,W
4,4,AL011851,UNNAMED,1851-06-25 21:00:00,L,HU,28.20,-96.80,80,NaN,...,-1.10,1.07,0.60,-0.47,69.22,77.11,20.73,20.73,89.95,W
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52686,52686,AL302020,THETA,2020-11-15 12:00:00,NaN,LO,31.40,-18.30,30,1009.00,...,0.32,0.30,0.51,0.21,-139.50,-106.26,87.60,87.60,-51.90,E
52687,52687,AL302020,THETA,2020-11-15 18:00:00,NaN,LO,31.60,-18.00,30,1012.00,...,0.94,0.51,0.96,0.45,-51.90,-139.50,42.29,42.29,-9.62,E
52688,52688,AL302020,THETA,2020-11-16 00:00:00,NaN,LO,32.10,-17.90,30,1013.00,...,0.74,0.96,1.21,0.25,-9.62,-51.90,32.42,32.42,22.81,E
52689,52689,AL302020,THETA,2020-11-16 06:00:00,NaN,LO,32.70,-18.20,25,1014.00,...,0.64,1.21,1.39,0.18,22.81,-9.62,2.68,2.68,25.48,W


In [9]:
edited_atlantic_storms_df.to_csv('edited_atlantic_storms.csv', index = False)